In [0]:
!pip install kagglehub[pandas-datasets]

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import shutil
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [0]:
dataset_files = [
    "LI-Small_Trans.csv",
    "LI-Small_accounts.csv",
    "LI-Small_Patterns.txt"
]

volume_base = "/Volumes/ibm-aml/bronze/raw-store/"
dataset_name = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"

for fname in dataset_files:
    local_path = kagglehub.dataset_download(dataset_name, path = fname)

    print(f"dataset file: {fname} downloaded into {local_path}")

    dest_path = os.path.join(volume_base, fname)
    
    print(f"Moving {fname} to Volume...")
    shutil.copy(local_path, dest_path)

## Read Downloaded files as Delta Tables

Since databricks offers its distributed processing capabilities through the Delta table abstractions we will use them to their fullest power.

In [0]:
from pyspark.sql.types import ( StructType, StructField, StringType, DecimalType, TimestampType, IntegerType, LongType ) 

# 1. Schema for LI-Medium_Trans.csv 
transactions_schema = StructType(
    [
        StructField("timestamp", TimestampType(), True), 
        StructField("from_bank", StringType(), True), 
        StructField("from_account", StringType(), True), 
        StructField("to_bank", StringType(), True), 
        StructField("to_account", StringType(), True), 
        StructField("amount_received", DecimalType(18, 2), True), 
        StructField("receiving_currency", StringType(), True), 
        StructField("amount_paid", DecimalType(18, 2), True), 
        StructField("payment_currency", StringType(), True), 
        StructField("payment_format", StringType(), True),
        StructField("is_laundering", IntegerType(), True)
    ])

# 2. Schema for LI-Medium_accounts.csv 
accounts_schema = StructType(
    [
        StructField("bank_name", StringType(), True), 
        StructField("bank_id", StringType(), True), 
        StructField("account_number", StringType(), True), 
        StructField("entity_id", StringType(), True), 
        StructField("entity_name", StringType(), True)
    ])

In [0]:
TRANSACTIONS_CSV_PATH = os.path.join(volume_base, dataset_files[0])
ACCOUNTS_CSV_PATH = os.path.join(volume_base, dataset_files[1])

# Ingest Transactions with fixed schema 
df_transactions = (
    spark.read.format("csv")
    .option("header", "true") 
    .option("timestampFormat", "yyyy/MM/dd HH:mm") 
    .schema(transactions_schema) 
    .load(TRANSACTIONS_CSV_PATH) 
    ) 
# Ingest Accounts with fixed schema 
df_accounts = (
    spark.read.format("csv") 
    .option("header", "true") 
    .schema(accounts_schema) 
    .load(ACCOUNTS_CSV_PATH) 
    ) 
# Save to Bronze Layer Delta Tables 
(
    df_transactions.write.format("delta") 
    .mode("overwrite") 
    .saveAsTable("`ibm-aml`.bronze.aml_bronze_transactions")
) 

(
    df_accounts.write
    .format("delta") 
    .mode("overwrite") 
    .saveAsTable("`ibm-aml`.bronze.aml_bronze_accounts") 
)

In [0]:
df_trans = spark.table("`ibm-aml`.bronze.aml_bronze_transactions_small") 
df_accounts = spark.table("`ibm-aml`.bronze.aml_bronze_accounts_small")

In [0]:
df_trans.columns

In [0]:
display(df_trans.select("payment_currency").distinct())